### t-SNE Embeddings

In [5]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.manifold import TSNE

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


In [7]:
dataset = load_dataset('surrey-nlp/BESSTIE-CW-26')

In [8]:
test_df = pd.DataFrame(dataset['test'])
test_df['Sarcasm'] = test_df['Sarcasm'].astype(int)

In [10]:
# load roberta
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
model = AutoModelForSequenceClassification.from_pretrained('models/roberta_sarcasm')
model = model.to(device)
model.eval()

print("Model loaded!")
print("Test size:", len(test_df))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  is thrown and the `use_auth_token` value is ignored.


Model loaded!
Test size: 2183


In [11]:
def get_embeddings(texts, batch_size=32):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        encodings   = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors='pt'
        )
        
        input_ids      = encodings['input_ids'].to(device)
        attention_mask = encodings['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model.roberta(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            # take CLS token embedding
            cls_embedding = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_embedding.cpu().numpy())
    return np.vstack(all_embeddings)

embeddings = get_embeddings(test_df['text'].tolist())
print("Embeddings shape: ", embeddings.shape)

Embeddings shape:  (2183, 768)


In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(embeddings)
print(f"")